# Create The Brain Prize Awards (PRIZE PATTERN)

The Brain Prize laureates from the official Brain Prize / Lundbeck Foundation Drupal pages.

**Prerequisite:** run `scripts/local/brain_prize_to_s3.py` to fetch the official winners page, yearly pages, laureate profile pages, verify the official amount rule page, and upload parquet to S3.

**Source:** `https://brainprize.org/winners`, yearly pages linked from that official page, laureate profile pages linked from each yearly page, and `https://brainprize.org/about-the-brain-prize` for the amount rule.

**S3 location:** `s3://openalex-ingest/awards/brain_prize/brain_prize_laureates.parquet`

**OpenAlex funder:** Lundbeckfonden, `funder_id = 4320321999`, ROR `https://ror.org/03hz8wd80`, DOI `10.13039/501100003554`.

**Schema notes:**
- Prize pattern: one row per official laureate record.
- `lead_investigator` is the laureate.
- `funding_type = 'prize'`.
- Provenance is `brain_prize`.
- The official About page says recipients share `DKK 10m`; the raw data preserves `source_total_award_amount`, `laureate_count`, and `portion`, and this notebook maps per-laureate `amount = source_total_award_amount / laureate_count`.
- Affiliation text is preserved from official laureate pages. Country is not separately mapped because the source does not publish a structured country field for every laureate.

**Contractor handoff:** local validation was completed without AWS/S3 or Databricks credentials. Admin must upload the parquet, run this notebook, run QA, and then run `CreateAwards.ipynb`. No job YAML is included until admin verification.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.brain_prize_raw
USING delta AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/brain_prize/brain_prize_laureates.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.brain_prize_raw;


## Step 1.5: Inspect Raw Data First

Verify exact source columns and values before relying on the transform below. The source script writes all raw columns as strings with `df.astype("string")` before parquet.


In [ ]:
%sql
DESCRIBE openalex.awards.brain_prize_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.brain_prize_raw
LIMIT 10;


In [ ]:
%sql
-- Money-shaped columns from the raw source. The yearly total amount is
-- verified from the official About page and split by laureate_count below.
SELECT column_name
FROM (DESCRIBE openalex.awards.brain_prize_raw)
WHERE LOWER(column_name) RLIKE 'amount|currency|fund|award|value|money|prize|dkk|portion|laureate_count';


In [ ]:
%sql
SELECT
    currency,
    COUNT(*) AS rows,
    COUNT(source_total_award_amount) AS rows_with_total_amount,
    COUNT(laureate_count) AS rows_with_laureate_count,
    MIN(TRY_CAST(source_total_award_amount AS DOUBLE)) AS min_total_amount,
    MAX(TRY_CAST(source_total_award_amount AS DOUBLE)) AS max_total_amount,
    MIN(TRY_CAST(laureate_count AS INT)) AS min_laureate_count,
    MAX(TRY_CAST(laureate_count AS INT)) AS max_laureate_count,
    collect_set(amount_rule_url) AS amount_rule_urls
FROM openalex.awards.brain_prize_raw
GROUP BY currency
ORDER BY currency;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(funder_award_id) AS has_funder_award_id,
    COUNT(laureate_name) AS has_name,
    COUNT(award_year) AS has_year,
    COUNT(award_topic) AS has_topic,
    COUNT(affiliation) AS has_affiliation,
    COUNT(profile_description) AS has_profile_description,
    COUNT(year_description) AS has_year_description,
    COUNT(source_total_award_amount) AS has_total_amount,
    COUNT(currency) AS has_currency,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    MIN(TRY_CAST(award_year AS INT)) AS min_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_year
FROM openalex.awards.brain_prize_raw;


In [ ]:
%sql
SELECT
    award_year,
    award_topic,
    laureate_name,
    affiliation,
    source_total_award_amount,
    laureate_count,
    portion,
    currency,
    landing_page_url
FROM openalex.awards.brain_prize_raw
ORDER BY TRY_CAST(award_year AS INT) DESC, award_topic, laureate_name
LIMIT 50;


## Step 1.6: Funder Existence Fail-Fast

Must return exactly one row. If zero rows, stop before transforming because the `CROSS JOIN` would emit zero awards.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320321999;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.brain_prize_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321999
),
normalized AS (
    SELECT
        n.*,
        TRY_CAST(n.award_year AS INT) AS award_year_int,
        TRY_CAST(n.source_total_award_amount AS DOUBLE) AS total_amount_double,
        TRY_CAST(n.laureate_count AS INT) AS laureate_count_int
    FROM openalex.awards.brain_prize_raw n
),
awards AS (
    SELECT
        abs(xxhash64(CONCAT(
            TRY_CAST(f.funder_id AS STRING), ':brain-prize:', LOWER(n.funder_award_id)
        ))) % 9000000000 AS id,
        CONCAT('The Brain Prize ', TRY_CAST(n.award_year_int AS STRING), ' - ', n.award_topic, ' - ', n.laureate_name) AS display_name,
        COALESCE(NULLIF(n.year_description, ''), NULLIF(n.profile_description, '')) AS description,
        f.funder_id,
        n.funder_award_id,
        CASE
            WHEN n.total_amount_double IS NOT NULL AND n.laureate_count_int > 0
                THEN n.total_amount_double / n.laureate_count_int
            ELSE CAST(NULL AS DOUBLE)
        END AS amount,
        NULLIF(n.currency, '') AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'prize' AS funding_type,
        NULLIF(n.award_topic, '') AS funder_scheme,
        'brain_prize' AS provenance,
        TRY_TO_DATE(CONCAT(TRY_CAST(n.award_year_int AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
        TRY_TO_DATE(CONCAT(TRY_CAST(n.award_year_int AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
        n.award_year_int AS start_year,
        n.award_year_int AS end_year,
        struct(
            NULLIF(n.given_name, '') AS given_name,
            NULLIF(n.family_name, '') AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                NULLIF(n.affiliation, '') AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        n.landing_page_url,
        CAST(NULL AS STRING) AS doi,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM normalized n
    CROSS JOIN funder_resolved f
    WHERE n.funder_award_id IS NOT NULL
      AND n.award_year_int IS NOT NULL
      AND n.laureate_name IS NOT NULL
)
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G', TRY_CAST(id AS STRING)) AS works_api_url,
    created_date,
    updated_date
FROM awards;


## Step 3: Insert Into `openalex_awards_raw` With Priority 84

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'brain_prize' AND priority = 84;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    84 AS priority
FROM openalex.awards.brain_prize_awards;


## Verification

In [ ]:
%sql
SELECT COUNT(*) AS total
FROM openalex.awards.brain_prize_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.brain_prize_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_display_name,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_lead_investigator,
    COUNT(landing_page_url) AS has_landing_page_url,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.brain_prize_awards;


In [ ]:
%sql
SELECT
    funder.display_name AS funder_display_name,
    funder_id,
    COUNT(*) AS rows
FROM openalex.awards.brain_prize_awards
GROUP BY funder.display_name, funder_id;


In [ ]:
%sql
SELECT
    start_year,
    funder_scheme,
    currency,
    COUNT(*) AS rows,
    SUM(amount) AS total_amount,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount
FROM openalex.awards.brain_prize_awards
GROUP BY start_year, funder_scheme, currency
ORDER BY start_year DESC;


In [ ]:
%sql
-- Duplicate checks. Both should return zero rows.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.brain_prize_awards
GROUP BY id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.brain_prize_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT
    id,
    display_name,
    description,
    amount,
    currency,
    funder_scheme,
    start_year,
    lead_investigator,
    landing_page_url
FROM openalex.awards.brain_prize_awards
ORDER BY start_year DESC, funder_scheme, display_name
LIMIT 20;
